In [1]:
import pandas as pd
import spacy
import textstat
import collections
import matplotlib.pyplot as plt
import seaborn as sns
import string
from nltk.tokenize import word_tokenize
import numpy as np

nlp = spacy.load("en_core_web_sm", disable=["ner"])

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
def load_and_label(path, label, author, target_style):
    with open(path, 'r', encoding='utf-8') as f:
        # We assume files are split by double newlines based on previous steps
        text = f.read()
        paragraphs = [p.strip() for p in text.split('\n\n') if len(p.split()) > 20]
    
    return pd.DataFrame({
        'text': paragraphs,
        'label': label,         # Human, AI_Generic, AI_Imposter
        'author': author,       # Lovecraft, Austen
        'target_style': target_style # Which style was meant to be mimicked?
    })

# --- CONFIGURATION ---
files = [
    # Human (Class 1)
    ("../data/processed/cthulu.txt", "Human", "Lovecraft", "Lovecraft"),
    ("../data/processed/prideprejudice.txt", "Human", "Austen", "Austen"),
    
    # Generic AI (Class 2)
    ("../data/generated/cthulhu_class2.txt", "AI_Generic", "Gemini", "Lovecraft"),
    ("../data/generated/prideprejudice_class2.txt", "AI_Generic", "Gemini", "Austen"),

    # Imposter AI (Class 3)
    ("../data/generated/cthulhu_class3.txt", "AI_Imposter", "Gemini", "Lovecraft"),
    ("../data/generated/prideprejudice_class3.txt", "AI_Imposter", "Gemini", "Austen")
]

# Load and Combine
dfs = []
for path, label, author, target in files:
    try:
        df_chunk = load_and_label(path, label, author, target)
        dfs.append(df_chunk)
        print(f"Loaded {len(df_chunk)} paragraphs from {path}")
    except FileNotFoundError:
        print(f"WARNING: File not found - {path}")

df = pd.concat(dfs, ignore_index=True)
print(f"Total Dataset Size: {len(df)} paragraphs")

In [ ]:
def analyze_syntax(text):
    doc = nlp(text)
    
    # 1. POS Ratio (Adjectives / Nouns)
    pos_counts = doc.count_by(spacy.attrs.POS)
    adj = pos_counts.get(spacy.symbols.ADJ, 0)
    noun = pos_counts.get(spacy.symbols.NOUN, 1) # Safety 1 to avoid ZeroDiv
    pos_ratio = adj / noun
    
    # 2. Dependency Tree Depth
    # Average depth of all sentences in the paragraph
    depths = []
    for sent in doc.sents:
        roots = [token for token in sent if token.head == token]
        if not roots: continue
        
        # Recursive depth finder
        def get_depth(node):
            if not list(node.children): return 1
            return 1 + max(get_depth(child) for child in node.children)
        
        depths.append(get_depth(roots[0]))
    
    avg_depth = np.mean(depths) if depths else 0
    
    # 3. Readability
    flesch = textstat.flesch_kincaid_grade(text)
    
    return pd.Series([pos_ratio, avg_depth, flesch], 
                     index=['adj_noun_ratio', 'avg_tree_depth', 'flesch_kincaid'])

# Apply to the dataframe (This might take a minute)
print("Running SpaCy Analysis... this may take time...")
df[['adj_noun_ratio', 'avg_tree_depth', 'flesch_kincaid']] = df['text'].apply(analyze_syntax)
print("Done.")

In [ ]:
def get_corpus_metrics(text_list, sample_size=5000):
    """
    Joins paragraphs into one giant text, takes a 5000-word slice,
    and calculates Hapax and TTR.
    """
    # Join all text and tokenize
    full_text = " ".join(text_list).lower()
    tokens = [t.text for t in nlp(full_text) if t.is_alpha]
    
    # Slice to exactly sample_size (if available)
    if len(tokens) > sample_size:
        tokens = tokens[:sample_size]
    
    # Calculations
    counts = collections.Counter(tokens)
    
    # TTR: Unique types / Total tokens
    ttr = len(counts) / len(tokens)
    
    # Hapax: Words appearing exactly once
    hapax = sum(1 for word, count in counts.items() if count == 1)
    
    return ttr, hapax

# Group by Class and Target Style to compare apples-to-apples
results = []
for (label, target), group in df.groupby(['label', 'target_style']):
    ttr, hapax = get_corpus_metrics(group['text'].tolist())
    results.append({
        'Label': label,
        'Target': target,
        'TTR': ttr,
        'Hapax_Legomena': hapax
    })

df_lexical = pd.DataFrame(results)
print(df_lexical)

In [ ]:
# 1. Adjective/Noun Ratio Plot
plt.figure(figsize=(12, 5))
sns.boxplot(x='target_style', y='adj_noun_ratio', hue='label', data=df)
plt.title("Adjective / Noun Ratio: Does AI 'Over-Describe'?")
plt.show()

# 2. Tree Depth Plot
plt.figure(figsize=(12, 5))
sns.boxplot(x='target_style', y='avg_tree_depth', hue='label', data=df)
plt.title("Syntactic Complexity (Tree Depth)")
plt.show()